# **PopOut — Estratégias de Pesquisa Adversarial e Árvores de Decisão**
### Inteligência Artificial 2025/2026

| | |
|---|---|
| **Grupo** | TP 6 Grupo 6 |
| **Elementos** | Eduardo Moura — nº202406710 · Filipe Huang — nº202406540 · Diego Nóbrega — nº202407575 |
---

## Índice
1. [Introdução](#1-introdução)
2. [O Jogo PopOut](#2-o-jogo-popout)
3. [Monte Carlo Tree Search (MCTS)](#3-monte-carlo-tree-search-mcts)
4. [Geração do Dataset](#4-geração-do-dataset)
5. [Árvore de Decisão — ID3](#5-árvore-de-decisão--id3)
6. [Computador vs. Computador](#6-computador-vs-computador)
7. [Resultados e Discussão](#7-resultados-e-discussão)
8. [Conclusão](#8-conclusão)


---
## **1. Introdução**

Este trabalho tem como objetivo implementar e avaliar dois agentes de Inteligência Artificial capazes de jogar **PopOut**, uma variante do *Connect-4*:

- **MCTS** (*Monte Carlo Tree Search*) — algoritmo de pesquisa adversarial que estima a qualidade de cada jogada através de simulações aleatórias.
- **ID3** (*Iterative Dichotomiser 3*) — algoritmo de aprendizagem supervisionada que constrói uma árvore de decisão a partir de um dataset gerado pelo MCTS.

### Restrições consideradas
- Não foram utilizadas bibliotecas de aprendizagem automática (e.g., `scikit-learn`) para treinar ou definir as árvores de decisão.
- O dataset de treino do ID3 foi gerado internamente através de simulações MCTS.
- Toda a lógica de jogo, pesquisa e aprendizagem foi implementada de raiz.

---
## **2. O Jogo PopOut**

A maior diferença entre o **PopOut** e o jogo base **Connect 4** é a possibilidade de fazer o movimento ***pop*** (remover a peça do fundo de uma coluna, consequentemente fazendo com que todas as peças acima descem um nível).

Além disso, tem 3 regras importantes a ter em consideração:
- 1: se o movimento ***pop*** criar 4 em linha para amobs os jogadores vence o jogador que efetuou a jogada.
- 2: quando o tabuleiro encontra-se totalmente preenchido, o jogador atual tem a hipótese de decidir se quer empatar o jogo ou não.
- 3: se um estado de tabuleiro repete-se mais de 3 vezes, qualquer jogador poder empatar a partida.

Todas as regras retantes são identicas ao **Connect 4**.

### **2.1.1 Representação de dados e inicialização**

In [ ]:
self.board = [[0]*7 for _ in range(6)]
self.player = 1
self.history = {}
self.last_move_pop = False
self.draw_state = False

- **board**: matiz 6x7(linhas x clounas), onde ``0=vazio``, ``1=player1`` e ``2=player2``.
- **player**: 1('X') ou 2('O'), indica o jogador atual.
-**history**: dicinário que mapeia os hashes do tabuleiro(freqência), usado para detetar repetições do estado do tabuleiro(útil para a regra 3 assim identificado).
- **last_move_pop**: se o ***pop*** foi a condição de vitória para ambos os jogadores na mesma jogada, esta expessão booleana é verificada.
- **draw_state**: expressão booleana que verifica se há possibilidades de empate ou não.

### **2.1.2 Copy**

In [ ]:
    def copy(self):
        new = PopOutState()
        new.board = [row[:] for row in self.board]
        new.player = self.player
        new.history = dict(self.history)
        new.last_move_pop = self.last_move_pop
        new.draw_state = self.draw_state
        return new

A função ``copy`` é essencial para algoritmos de pesquisa com *Monte Carlo Tree Search*, para explorar caminhos sem modificar o estado original.

### **2.1.3 Geração de Movimentos**


In [ ]:
def get_legal_moves(self):
    moves = []
    for c in range(7):
        if any(self.board[r][c]==0 for r in range(6)): moves.append((c, 'put'))
        if self.board[0][c] == self.player: moves.append((c, 'pop'))
    if self.is_full(): moves.append((-1, 'draw'))
    return moves

Basea-se principalmente em 3 movimentos:
- ``put`` ou *drop*: por a peça - possível se certa coluna não estiver totalmente preenchida, ou seja, sempre que haja posições com estado igual a *0*.
- ``pop``: retirar uma peça de baixo - só é permitido se a peça pertence ao jogador atual.
- ``draw``: empatar o jogo - somente se uma das regras de 2 e 3 for verificada.

### **2.1.4 Execução de Movimento**

In [ ]:
def make_move(self, moves):
    col, action = moves
    new_state = self.copy()
    new_state.last_move_pop = False
    if action == 'draw':
        new_state.draw_state = True
        new_state.player = 3-self.player
        new_state.update_history()
        return new_state
    if action == 'pop':
        for r in range(5): new_state.board[r][col] = new_state.board[r+1][col]
        new_state.board[5][col] = 0
        new_state.last_move_pop = True
    else:  # put
        for r in range(6):
            if new_state.board[r][col] == 0:
                new_state.board[r][col] = new_state.player
                break
    new_state.player = 3 - self.player
    new_state.update_history()
    return new_state

A função ``make_move`` retorna um novo estado após de realizar o movimento(*put*, *pop*, *draw*) sem mexer no original.

O estado é guardado através de ``update_history`` para poder detetar repetições de estados(regara 3).

**Nota**: o *pop* neste função não verifica se a peça retirada pertence ou não ao próprio pois já é verificada no ``get_legal_move``.

### **2.1.5 Condição Terminal e Deteção de Vitória**

In [ ]:
def get_winner(self):
    if self.draw_state: return 'draw'
    if any(count>=3 for count in self.history.values()): return 'draw'
    p1 = self.has_four_in_row(1)
    p2 = self.has_four_in_row(2)
    if p1 and p2: return 3 - self.player if self.last_move_pop else None
    if p1: return 1
    if p2: return 2
    return None

Condições de **empate**:
- por repetição
- por ter tabuleiro cheio

Condições de **vitória**:
- 4 em linha - horizontal, vertical ou diagonal
- vitória simultanea - ganha o jogador que realizou o movimento

### Interface de jogo
→ Implementada em `game/interface.py` — função `play_game()`

Suporta três modos:
1. **Humano vs. Humano**
2. **Humano vs. Computador** (MCTS ou ID3)
3. **Computador vs. Computador** (MCTS vs. ID3)


In [ ]:
def print_board(state):
    print("\n      0   1   2   3   4   5   6")
    print("   " + "─" * 29)
    for row in state.board[::-1]:
        line = " │ ".join(["X" if x == 1 else "O" if x == 2 else "-" for x in row])
        print("   │ " + line + " │")
    print("   " + "─" * 29)
    if not state.is_terminal(): print(f"   It is now {'X' if state.player == 1 else 'O'}'s turn.\n")

def play_game():
    while True:
        state = PopOutState()

        print("\n" + "=" * 35)
        print("           POPOUT GAME")
        print("=" * 35)
        print("Rules: Drop or Pop your pieces.\nFirst to 4 in a row wins!")
        print("Type 'put' or 'pop' when asked.\n")

        ai_mcts_player = None
        ai_id3_player = None

        while True:
            print("\nMain Menu:")
            print("(1) Human vs Human")
            print("(2) Human vs AI")
            print("(3) AI (MCTS) vs AI (ID3)")
            print("(4) Exit")
            mode = input("Choose mode (1/2/3/4): ").strip()
            if mode in ['1', '2', '3', '4']:
                break
            print("Please enter 1, 2, 3, or 4.")

        if mode == '4':
            print("Exiting game. Goodbye!")
            break

        if mode == '2':
            while True:
                print("\nChoose AI opponent:")
                print("(1) MCTS")
                print("(2) ID3")
                ai_choice = input("Choice (1/2): ").strip()
                if ai_choice in ['1', '2']:
                    break
                print("Please enter 1 or 2.")

            while True:
                symbol = input("Do you want to play as 'X' (first) or 'O' (second)? (X/O): ").strip().upper()
                if symbol in ['X', 'O']:
                    break
                print("Please enter X or O.")

            if symbol == 'X':
                if ai_choice == '1': ai_mcts_player = 2
                else: ai_id3_player = 2
            else:
                if ai_choice == '1': ai_mcts_player = 1
                else: ai_id3_player = 1

        elif mode == '3':
            while True:
                first = input("Who plays first as 'X'? (1) MCTS or (2) ID3: ").strip()
                if first in ['1', '2']:
                    break
                print("Please enter 1 or 2.")

            if first == '1':
                ai_mcts_player = 1
                ai_id3_player = 2
            else:
                ai_id3_player = 1
                ai_mcts_player = 2

        tree, features = None, None
        if ai_id3_player is not None:
            print("\nLoading dataset and training ID3 Tree... Please wait.")
            tree, features = train_tree(max_depth=10)
            if not tree:
                print("Warning: dataset.csv not found. ID3 will play randomly.")

        print_board(state)

        while not state.is_terminal():
            player_symbol = "X" if state.player == 1 else "O"
            print(f"\n{'=' * 35}")
            print(f"            {player_symbol}'S TURN")
            print(f"{'=' * 35}")

            moves = state.get_legal_moves()

            if ai_mcts_player and state.player == ai_mcts_player:
                print("MCTS AI is thinking...")
                move = mcts_search(state, iterations=1000)
                print(f"MCTS played: column {move[0]}, {move[1]}")
                state = state.make_move(move)
                print_board(state)
                continue

            if ai_id3_player and state.player == ai_id3_player:
                print("ID3 AI is thinking...")
                if tree:
                    move = get_id3_move(state, tree, features)
                    if move not in moves:
                        move = moves[0]
                else:
                    import random
                    move = random.choice(moves)

                print(f"ID3 played: column {move[0]}, {move[1]}")
                state = state.make_move(move)
                print_board(state)
                continue

            print(f"Legal moves: {moves}")

            is_draw = any(m[1]=='draw' for m in moves)

            if is_draw: text = "\nEnter column (0-6) or 'd' for DRAW: "
            else: text = "\nEnter column (0-6): "

            while True:
                try:
                    choice = input(text)
                    if choice=='d':
                        if is_draw:
                            state = state.make_move((-1, 'draw'))
                            break
                        else: continue
                    col = int(choice)
                    if not 0 <= col <= 6:
                        print("Column must be between 0 and 6!")
                        continue

                    can_put = any(state.board[r][col] == 0 for r in range(6))
                    can_pop = (state.board[0][col] == state.player)

                    if not can_put and not can_pop:
                        print("No moves possible in this column.")
                        continue
                    if can_put and can_pop:
                        ask = input("Enter action ('+' for put or '-' for pop): ").strip().lower()
                        if ask not in ['+', '-']:
                            print("Please type '+' or '-'.")
                            continue
                        action = 'pop' if ask == '-' else 'put'
                    elif can_put:
                        print(f"Only 'put' is possible in column {col}.")
                        action = 'put'
                    else:
                        print(f"Only 'pop' is possible in column {col}.")
                        action = 'pop'

                    move = (col, action)
                    if move in moves:
                        state = state.make_move(move)
                        break
                    else: print("This move is not legal.")
                except ValueError: print("Please enter a valid number for column.")

            print_board(state)

        #Game Over
        print("\n" + "=" * 35)
        winner = state.get_winner()
        if winner == "draw": print("             GAME DRAW!")
        elif winner == 1: print("             'X' WINS!")
        elif winner == 2: print("             'O' WINS!")
        print("=" * 35)

---
## **3. Monte Carlo Tree Search (MCTS)**

O MCTS é um algoritmo de **pesquisa adversarial** que constrói iterativamente uma árvore
orientada por simulações aleatórias (*rollouts*). Cada iteração passa por 4 fases:

| Fase | Descrição |
|---|---|
| **1. Seleção** | Percorre a árvore usando UCT até encontrar um nó não totalmente expandido |
| **2. Expansão** | Adiciona um novo nó filho para uma jogada ainda não explorada |
| **3. Simulação** | Executa um *rollout* aleatório até ao estado terminal |
| **4. Retropropagação** | Propaga o resultado para todos os nós ancestrais |

### Critério de seleção — UCT (*Upper Confidence Bound for Trees*)

$$UCT(i) = \frac{w_i}{n_i} + c \cdot \sqrt{\frac{\ln N}{n_i}}$$

Onde $w_i$ = vitórias, $n_i$ = visitas ao nó $i$, $N$ = visitas ao pai, $c$ = constante de exploração ($\sqrt{2} \approx 1.41$).

→ Implementado em `game/mcts.py` — classe `MCTSNode`, função `mcts_search()`


In [ ]:
class MCTSNode:
    def __init__(self, state, parent=None, move=None):
        self.state = state
        self.parent = parent
        self.move = move
        self.children = []
        self.wins = 0
        self.visits = 0
        self.untried_moves = state.get_legal_moves()

    def is_fully_expanded(self):
        return len(self.untried_moves) == 0

    def best_child(self, c=1.41):
        return max(self.children, key=lambda n:
            n.wins / n.visits + c * math.sqrt(math.log(self.visits) / n.visits)
        )

    def expand(self):
        move = self.untried_moves.pop()
        new_state = self.state.make_move(move)
        child = MCTSNode(new_state, parent=self, move=move)
        self.children.append(child)
        return child

    def rollout(self):
        state = self.state.copy()
        while not state.is_terminal():
            moves = state.get_legal_moves()
            state = state.make_move(random.choice(moves))
        return state.get_winner()

    def backpropagate(self, result, ai_player):
        self.visits += 1
        if result == 'draw':
            self.wins -= 0.5
        elif result == ai_player:
            self.wins += 1
        else: self.wins -= 15
        if self.parent:
            self.parent.backpropagate(result, ai_player)


def mcts_search(state, iterations=1500, c=1.41):
    total_pieces = sum(cell != 0 for row in state.board for cell in row)
    if total_pieces < 2:
        moves = state.get_legal_moves()
        return random.choice(moves)

    ai_player = state.player
    root = MCTSNode(state)

    for _ in range(iterations):
        node = root
        while node.is_fully_expanded() and node.children:
            node = node.best_child(c)

        if not node.is_fully_expanded():
            node = node.expand()

        result = node.rollout()

        node.backpropagate(result, ai_player)

    best = max(root.children, key=lambda n: n.visits)
    return best.move

### Variantes exploradas

| Variante | Parâmetro alterado | Objetivo |
|---|---|---|
| **MCTS Padrão** | `iterations=1000`, `c=1.41` | Baseline de referência |
| **Exploração agressiva** | `c=2.0`, `c=2.5` | Favorece nós pouco visitados |
| **Exploração conservadora** | `c=0.5`, `c=0.8` | Favorece nós com bom histórico |

**Início aleatório**: Primeiras 2 peças aleatórias (Aumenta variedade do jogo)

Os resultados comparativos estão na Secção 7.


---
## **4. Geração do Dataset com Features Estratégicas**

Em vez de alimentar o ID3 com as 42 células em bruto (que leva a árvores gigantes e overfitting), optamos por **extrair features de alto nível** que capturam a essência da posição no PopOut.

Este pré‑processamento melhora a generalização e reduz drasticamente a dimensionalidade.

### **4.1 Features definidas**

Todas as features são calculadas a partir do tabuleiro e do jogador atual (`p`) e do oponente (`opp = 3 - p`). São maioritariamente discretizadas em três categorias (`"low"`, `"mid"`, `"high"`) ou limitadas a pequenos inteiros.

| Feature | Descrição | Discretização / Domínio |
|---------|-----------|-------------------------|
| `center_adv` | Diferença de peças nas três colunas centrais(2,3,4) entre o jogador e o oponente |  `low` / `mid` / `high` |
| `threats_me_3` / `threats_opp_3` | Nº de sequências de **exatamente 3 peças** consecutivas(fechadas ou abertas) | `min(contagem, 4)`|
| `open_threats_me` / `open_threats_opp` | Sequências de **3 peças com pelo menos uma célula vazia adjacente**(ameaça real de 4 em linha na próxima jogada) | `min(threats(board, player), 4)` |
| `pairs_me` / `pairs_opp` | Número de pares (2 peças consecutivas) | `min(contagem, 6)` |
| `pop_me` / `pop_opp` | No de colunas onde a linha 0 pertence ao jogador(possibilidade de *pop*) | valor bruto (0‑7) |
| `col_h_0` … `col_h_6` | Altura de cada coluna(0 a 6) | `_bin3(h, 0, 6)` |
| `my_col_0` … `my_col_6` | Quantas peças próprias em cada coluna | `min(contagem, 4)` |
| `opp_col_0` … `opp_col_6` | Quantas peças do oponente em cada coluna | `min(contagem, 4)` |
| `phase` | Fase do jogo com base no número total de peças no tabuleiro | `"early"` (≤8), `"mid"` (≤24), `"late"` (>24) |
| `lr_balance` | Diferença (esquerda – direita) de peças próprias nas três colunas da esquerda (0‑2) vs. três da direita (4‑6) | `_bin3(diff, -6, 6)` |

### **4.2 Funções auxiliares**

- **`count_consecutive`**  
  Conta todas as sequências exatas de `length` peças consecutivas em qualquer direção. Utilizada para `threats_*_3` e `pairs_*`.

- **`threats`**  
  Conta sequências de `length` peças que têm **pelo menos uma célula vazia adjacente** nas extremidades, ou seja, ameaças reais de completar 4 em linha na jogada seguinte(caso o espaço vazio seja preenchido).

- **`col_height`**  
  Altura preenchida da coluna(nº de peças desde o fundo).

- **`center_control`**  
  Nº de peças do jogador nas colunas centrais 2, 3 e 4. O domínio varia entre 0 e 18(6 linhas x 3 colunas), daí os limites `-9` a `+9` na diferença.

- **`poppable_pieces`**  
  No de colunas onde a base pertence ao jogador – corresponde às colunas onde é possível fazer *pop*.

- **`_bin3`**  
  Discretiza um valor contínuo(ou inteiro) em três categorias:  
  - `"low"` se `value < lo+(hi-lo)/3`  
  - `"mid"` se `value < lo+2(hi-lo)/3`  
  - `"high"` caso contrário.  
  Isto transforma features numéricas em atributos nominais, necessários para o ID3.

### **4.3 Transformação do estado (`state_to_features`)**

A função `state_to_features(state)` recebe um objeto `PopOutState` e devolve um **dicionário** com todas as features descritas acima. Exemplo de saída:

```python
{
    'center_adv': 'mid',
    'threats_me_3': 2,
    'threats_opp_3': 1,
    'open_threats_me': 1,
    'open_threats_opp': 0,
    'pairs_me': 3,
    'pairs_opp': 2,
    'pop_me': 2,
    'pop_opp': 1,
    'col_h_0': 'low',
    ...,
    'phase': 'early',
    'lr_balance': 'high',
    'move': '3_put'
}
```

### **4.4 Geração do dataset**

O dataset é criado através de **auto‑jogo MCTS(MCTS vs. MCTS)**. Para cada posição encontrada durante os jogos, o MCTS(com ``iterations=2000``) determina a melhor jogada, que serve como *label*.

In [ ]:
def generate_dataset(num_games=1000, iterations=2000, output_file="dataset.csv"):
    header = list(state_to_features(PopOutState()).keys()) + ["move"]
    seen_states = set()
    with open(output_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=header)
        writer.writeheader()
        for game_num in range(num_games):
            state = PopOutState()
            while not state.is_terminal():
                best_move = mcts_search(state, iterations=iterations, c=20)
                feats = state_to_features(state)
                # Evita duplicados: usa o dicionário ordenado como chave
                state_key = tuple(sorted(feats.items()))
                if state_key not in seen_states:
                    seen_states.add(state_key)
                    col, action = best_move
                    label = f"{col}_{action}"
                    row = {**feats, "move": label}
                    writer.writerow(row)
                    total += 1
                state = state.make_move(best_move)

### **4.5 Vantagens em relação à representação bruto**

| Critério | Células brutas | Features estratégicas |
|-|-|-|
| Dimencionalidade | 43 | 32 |
| Generalização | Baixa | Ligeiramenta mais alta |
| Tamanho | Muito Grande | Moderado |

---
## **5. Árvore de Decisão — ID3 (Implementação e Melhorias)**

O **ID3** (*Iterative Dichotomiser 3*) é um algoritmo de aprendizagem supervisionada que constrói árvores de decisão a partir de dados, utilizando **entropia** e **ganho de informação** para escolher os atributos mais discriminativos.

**Entropia** de um conjunto $S$:
$$H(S) = -\sum_{c \in C} p_c \log_2 p_c$$
**Ganho de Informação** para o atributo $A$:
$$IG(S, A) = H(S)-\sum_{v\in valores(A)} \frac{|S_v|}{|S|} \cdot H(S_v)$$
> **Nota:** Sem uso de `scikit-learn` ou equivalentes. Implementação de raiz.

Para validar a nossa implementação antes de a aplicar ao jogo PopOut, testámos primeiro no conhecido dataset **Iris** (classificação de flores). De seguida, adaptámos e melhorámos o algoritmo para o nosso domínio, introduzindo **gain ratio**, **paragem por amostras mínimas** e **mecanismos de fallback** na predição.

### 5.1 Warm‑up: ID3 no dataset Iris

O dataset Iris tem atributos numéricos contínuos (comprimento/largura de sépalas e pétalas). Como o ID3 trabalha naturalmente com atributos discretos, foi necessário **discretizar** os valores.

#### Discretização por bins iguais

In [ ]:
def discretize(data, features, n_bins=3):
    step = (max_v - min_v) / n_bins
    thresholds[feat] = [min_v + step * i for i in range(1, n_bins)]
    bin_idx = sum(row[feat] > t for t in thresholds[feat])

- Utilizámos ``n_bins=3`` (três categorias: ``low``, ``mid``, ``high``).

- Os limites são calculados apenas a partir dos dados de treino para evitar data leakage.

- A mesma discretização é aplicada ao conjunto de teste (``discretize_row``).

#### **Entropia e Ganho de Informação**

In [ ]:
def entropy(data, label="class"):
    counts = Counter(row[label] for row in data)
    return -sum((c/n)*log2(c/n) for c in counts.values())
def information_gain(data, feature, label):
    base_entropy = entropy(data, label)
    weighted = sum((len(subset)/n)*entropy(subset, label)for v in values)
    return base_entropy - weighted

A construção da árvode (``id3``) segue o algoritmo clássico:

1. Se todos os exemplos têm a mesma classe → folha.
2. Se não há atributos ou profundidade máxima atingida → folha com classe maioritária.
3. Escolher atributo com maior ganho de informação.
4. Dividir os dados pelos valores desse atributo e recorrer.

**Resultados no iris**
- Treino: 80% (120 flores), teste: 20% (30 flores)
- Acurácia típica: >90%

Este warm-up foi para verificar se a implementação do ID3 está a funcionar corretamente.

### **5.2 ID3 aplicado ao PopOut**

Para o domínio do PopOut, decidimos não usar as 42 células brutas, mas sim features estratégicas (ver Secção 4). Estas features já são maioritariamente discretas. Assim, não é necessária discretização adicional.

No entanto, introduzimos várias melhorias face à versão Iris:

**a) Critério de seleção: Gain Ratio**

O ganho de informação puro tende a favorecer atributos com muitos valores

In [ ]:
def gain_ratio(data, feature, label="move"):
    ig = information_gain(data, feature, label)
    split_info = -sum((len(subset)/n) * log2(len(subset)/n) for v in values)
    return ig / split_info if split_info != 0 else 0

- Penaliza atributos que dispersam demasiado os dados
- Ajuda a evitar *overfitting*

**b) Critérios de paragem adicionais**

Al+em da profundidade máxima, adicionamos:

In [ ]:
if len(data) < min_samples: return Node(label=majority)

Isto impede a criação de folhas com muito poucos exemplos, que geralmente sao ruidos.

**c) Fallback robusto na predição**

Durante a predição, pode acontecer que um estado do jogo apresente um valor de feature não visto durante o treino. A nossa função predict lida com isso de duas formas:

In [ ]:
if val not in node.children:
    child_labels = _collect_labels(node)
    return Counter(child_labels).most_common(1)[0][0]

- Primeiro tenta recolher os labels de todos os sub‑nós e devolve o mais frequente.

- Em último caso, usa o ``majority_class`` guardado no nó(a classe mais comum do conjunto de treino naquele nó).

### **5.3 Diferenças chave entre as versões Iris e PopOut**

|Aspeto|Iris|PopOut|
|-|-|-|
|Atributos|Contínuos → discretização|Já discretos (features estratégicas)|
|Critério de divisão|Ganho de informação|Gain ratio|
|Paragem|Profundidade máxima|Profundidade+``min_samples``|
|Predição|Fallback simples (primeiro filho)|Voto maioritário dos filhos+``majority_class``|

---
## 6. Computador vs. Computador

Para comparar os dois agentes, foram realizados jogos automáticos MCTS vs. ID3.

→ Lógica de torneio em `game/interface.py` ou `game/tournament.py`


In [ ]:
# Torneio automático: MCTS vs. ID3
# Descomente após o ID3 estar integrado

# def torneio(agente_x, agente_o, n_jogos=50):
#     resultados = {'X (MCTS)': 0, 'O (ID3)': 0, 'Empate': 0}
#     for i in range(n_jogos):
#         state = PopOutState()
#         while not state.is_terminal():
#             move = agente_x(state) if state.player == 1 else agente_o(state)
#             state = state.make_move(move)
#         w = state.get_winner()
#         if w == 1:    resultados['X (MCTS)'] += 1
#         elif w == 2:  resultados['O (ID3)'] += 1
#         else:         resultados['Empate'] += 1
#         print(f"  Jogo {i+1}/{n_jogos} — vencedor: {w}", end='
')
#     return resultados

# resultados = torneio(
#     agente_x=lambda s: mcts_search(s, iterations=1000),
#     agente_o=lambda s: id3_move(s, tree_popout),
#     n_jogos=50
# )
# print("\nResultados:", resultados)

print("⚠️  Descomente após integrar o agente ID3")


In [ ]:
# Visualização dos resultados do torneio
# Preenche com os valores reais após o torneio

# resultados = {'X (MCTS)': 28, 'O (ID3)': 16, 'Empate': 6}  # substituir pelos reais

# import matplotlib.pyplot as plt
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# axes[0].bar(resultados.keys(), resultados.values(),
#             color=['#4e8cff', '#ff6b6b', '#aaa'], edgecolor='white', width=0.5)
# axes[0].set_title('MCTS (X) vs. ID3 (O) — 50 jogos')
# axes[0].set_ylabel('Número de vitórias')
# for i, (k, v) in enumerate(resultados.items()):
#     axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

# axes[1].pie(resultados.values(), labels=resultados.keys(), autopct='%1.1f%%',
#             colors=['#4e8cff', '#ff6b6b', '#aaa'])
# axes[1].set_title('Distribuição de resultados')

# plt.tight_layout()
# plt.show()

print("⚠️  Descomente após obter os resultados do torneio")


---
## 7. Resultados e Discussão

### 7.1 Resumo do desempenho

| Métrica | MCTS (1000 iter) | MCTS (3000 iter) | ID3 (PopOut) |
|---|---|---|---|
| Taxa de vitória vs. aleatório | X% | X% | X% |
| Taxa de vitória (MCTS vs. ID3) | X% | — | X% |
| Tempo médio por jogada | ~Xms | ~Xms | ~Xms |
| Qualidade da jogada | Alta | Muito alta | Depende do treino |
| Interpretabilidade | Baixa | Baixa | Alta |

*Preencher com valores reais após os experimentos.*

### 7.2 Efeito da constante de exploração $c$ no MCTS

| $c$ | Comportamento | Resultado observado |
|---|---|---|
| `0.5` | Mais exploração (nós com bom histórico) | [preencher] |
| `1.41` ($\sqrt{2}$, padrão) | Equilíbrio teórico exploração/exploração | [preencher] |
| `2.0` | Mais exploração (nós pouco visitados) | [preencher] |

### 7.3 Qualidade do ID3

- **Precisão no Iris**: X% (train/test 80/20)
- **Precisão no PopOut**: X%
- **Observações sobre o dataset**: [descrever distribuição, balanceamento, cobertura de estados]
- **Limitações**: O ID3 só conhece estados vistos durante a geração — estados raros podem gerar jogadas sub-ótimas.


---
## 8. Conclusão

Neste trabalho foram implementados com sucesso:
- O jogo **PopOut** com todas as regras, incluindo as 3 regras especiais
- Um agente **MCTS** com critério UCT, com análise do impacto do número de iterações e da constante $c$
- Uma árvore de decisão **ID3** de raiz, validada no Iris e aplicada ao PopOut
- Um **dataset** de X exemplos únicos gerado por auto-jogo MCTS (2000 iterações/jogada)
- Modo **Computador vs. Computador** para comparação direta dos dois agentes

### Reflexão comparativa teoricamente

O **MCTS** é mais forte porque avalia o estado em tempo real.
O **ID3** é mais rápido na inferência mas depende totalmente da qualidade e cobertura do dataset de treino.

### **Reflexao real dos resultados obtidos

### Possíveis melhorias
- Aumentar o dataset (mais jogos, mais variações)
- Usar profundidade máxima adaptativa no ID3
- Implementar *alpha-beta pruning* como terceiro agente para comparação
